# Fundamental Quality Score & Random Forest Model
Predict investment outcomes based on company fundamentals

## 1. Data collection and data preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

: 

In [ ]:
# Select company and load data
name = input("Enter the name of the company: ").upper()
df = pd.read_csv(f"../data-companywise/processed_data/{name}.csv")
df


In [ ]:
df

In [ ]:
# Check for missing values
df.isna().sum()

In [ ]:
df.dropna(axis =0, inplace = True)
df

In [ ]:
df.info()

## 2. Data Preprocessing

In [ ]:
# Set date as index for time-series analysis
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
df

In [ ]:
# Sort chronologically
df.sort_index(inplace=True)
df

In [ ]:
df.drop(columns = 'pe-ratio', inplace = True)
df

## 3. Feature Engineering

In [ ]:
#Get the first EPS value recorded for each year
annual_eps = df.groupby('year')['eps'].first()

#  Calculate the percentage change between those years
growth_values = annual_eps.pct_change().fillna(0)

#  Turn it into a map and apply it to the whole 'eps_growth' column
growth_map = growth_values.to_dict()
df['eps_growth'] = df['year'].map(growth_map).round(5)


In [ ]:
# Inverting the bad metrices which is considered lower as best
df['inv_pb'] = 1/ df['pb-ratio']
df['inv_de'] = df['debt-equity'].apply(lambda x: 1/x if x > 0.01 else 1.0) 

In [ ]:
#pe calculated from daily close price and annual EPS
df['pe_dynamic'] = df['close'] / df['eps']
df['inv_pe'] = 1/ df['pe_dynamic']

In [ ]:
df['pe_dynamic']

In [ ]:
df.isna().sum()

In [ ]:
#scaling using minmaxscaler from 0 - 1
df_known =   df.copy()

cols = ['roe', 'net-margin', 'eps_growth','inv_pe','inv_pb','dividend', 'inv_de']
scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_known[cols]), columns = cols, index = df_known.index)
df_scaled

In [ ]:
df['inv_pe']

## 4. Quality score Calculation

In [ ]:
# Grouping into the categories
profitability = ['roe', 'net-margin', 'eps_growth']
valuation = ['inv_pe', 'inv_pb'] 
safety = ['dividend', 'inv_de']
df_known['quality_score'] = (
    df_scaled[profitability].mean(axis=1) * 0.50 + 
    df_scaled[valuation].mean(axis=1) * 0.30 + 
    df_scaled[safety].mean(axis=1) * 0.20
).round(4)



In [ ]:
df_known['quality_score'].head(5)

## 5. Visualizing the Quality score Trend

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df_known.index, df_known['quality_score'], color='green', linewidth=1.5)
plt.title('Daily Quality Score From 2022 to 2025)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Quality Score (0–1)')
plt.grid(axis='y', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig('../results/plots/quality_score_trend.png', dpi=300)
plt.show()

## 6. Random Forest Model Training

In [ ]:
from sklearn.ensemble import RandomForestRegressor

### i. Defining the features for the model.

In [ ]:
#creating the lag features from past quality score
df_known['qs_lag1'] = df_known['quality_score'].shift(1) #from previous day
df_known['qs_lag2'] = df_known['quality_score'].shift(2)
df_known['qs_lag3'] = df_known['quality_score'].shift(3)

#price momentum
df_known['return_1d']  = df_known['close'].pct_change(1)   
df_known['return_5d']  = df_known['close'].pct_change(5)  
df_known['return_20d'] = df_known['close'].pct_change(20)

#moving average
df_known['ma7']  = df_known['close'].rolling(7).mean()
df_known['ma21'] = df_known['close'].rolling(21).mean()
df_known['ma_ratio'] = df_known['ma7'] / df_known['ma21']

#targetting the next day quality score.
df_known['qs_next'] = df_known['quality_score'].shift(-1)


In [ ]:
#dropping nan
print(df_known.isna().sum())

df_model = df_known.dropna(subset=['qs_lag1','qs_lag2','qs_lag3','return_1d',
'return_5d','return_20d','ma_ratio','qs_next'])

In [ ]:
print(f"Total rows for modelling: {len(df_model)}")
print(df_model[['qs_lag1','qs_lag2','qs_lag3','return_1d',
'return_5d','return_20d','ma_ratio','qs_next']].head())

### ii. Preparing the training set

In [ ]:
features = ['qs_lag1','qs_lag2','qs_lag3','return_1d','return_5d','return_20d','ma_ratio','qs_next']

X_train = df_model.loc['2022':'2024', features]
y_train = df_model.loc['2022':'2024', 'qs_next']

X_test  = df_model.loc['2025', features]
y_test  = df_model.loc['2025', 'qs_next']

print(f"Train size: {len(X_train)}")
print(f"Test size:  {len(X_test)}")

### iii. Scaling the data using MinMaxscaler and model fitting

In [ ]:
scaler_rf = MinMaxScaler()
X_train_scaled = scaler_rf.fit_transform(X_train)
X_test_scaled  = scaler_rf.transform(X_test)   # same scaler, only transform

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

print("Model trained successfully.")

### iv. predicting the outcome

In [ ]:
y_pred = model.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

actual_dir    = np.sign(np.diff(y_test.values))
predicted_dir = np.sign(np.diff(y_pred))
directional_acc = np.mean(actual_dir == predicted_dir)

print(f"RMSE:{rmse:.4f}")
print(f"R²:{r2:.4f}")
print(f"Directional Accuracy: {directional_acc:.2%}")

### v. Checking for the type of signal

In [ ]:
latest_qs = df_known['quality_score'].iloc[-1]

print(f"\nLatest 2026 Quality Score: {latest_qs:.4f}")

if latest_qs >= 0.6:
    print("Signal: BULLISH (Buy)")
elif latest_qs <= 0.4:
    print("Signal: BEARISH (Sell)")
else:
    print("Signal: NEUTRAL (Hold)")

### vi. Considering the important features.

In [ ]:
importances = model.feature_importances_
feat_df = pd.DataFrame({
    'feature': features,
    'importance': importances
}).sort_values('importance', ascending=True)

feat_df.plot(kind='barh', x='feature', y='importance',
             figsize=(8,5), legend=False,
             title='RF Feature Importance')
plt.tight_layout()
plt.savefig('../results/plots/feature_importance.png', dpi=300)
plt.show()

## 7. Visualizing the result

### Plot 1 — QS Trend with Buy/Sell/Hold Zones

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df_known.index, df_known['quality_score'], color='steelblue', linewidth=1.2, label='Quality Score')

# Zone bands
plt.axhspan(0.7, 1.0, alpha=0.15, color='green', label='Bullish Zone (≥0.7)')
plt.axhspan(0.4, 0.7, alpha=0.15, color='orange', label='Neutral Zone')
plt.axhspan(0.0, 0.4, alpha=0.15, color='red', label='Bearish Zone (≤0.4)')

plt.axhline(0.7, color='green', linestyle='--', linewidth=0.8)
plt.axhline(0.4, color='red', linestyle='--', linewidth=0.8)

plt.title('Daily Quality Score with Signal Zones', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Quality Score')
plt.legend(loc='upper left')
plt.tight_layout()
plt.savefig('../results/plots/qs_zones.png', dpi=300)
plt.show()

### Plot 2 — Correlation Heatmap

In [ ]:
import seaborn as sns

# Features used in QS calculation + quality score itself
corr_cols = [
    'roe', 'net-margin', 'eps_growth','inv_pe', 'inv_pb','dividend', 'inv_de','quality_score'
]

corr_matrix = df_known[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)

plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/correlation_heatmap.png', dpi=300)
plt.show()

### Plot 3 — Yearly Average QS (Bar Chart)

In [ ]:
yearly_qs = df_known.groupby('year')['quality_score'].mean()

colors = ['green' if v >= 0.6 else 'red' if v <= 0.4 else 'orange' for v in yearly_qs]

plt.figure(figsize=(8, 5))
bars = plt.bar(yearly_qs.index.astype(str), yearly_qs.values, color=colors, edgecolor='black', width=0.5)

for bar, val in zip(bars, yearly_qs.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.axhline(0.6, color='green', linestyle='--', linewidth=1, label='Bullish threshold')
plt.axhline(0.4, color='red',   linestyle='--', linewidth=1, label='Bearish threshold')

plt.title('Yearly Average Quality Score', fontsize=14, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Average Quality Score')
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.savefig('../results/plots/qs_yearly_avg.png', dpi=300)
plt.show()

### Plot 4 — QS Distribution (Histogram)

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df_known['quality_score'], bins=40, color='steelblue', edgecolor='white')

plt.axvline(0.6, color='green', linestyle='--', linewidth=1.5, label='Bullish threshold (0.7)')
plt.axvline(0.4, color='red',   linestyle='--', linewidth=1.5, label='Bearish threshold (0.4)')
plt.axvline(df_known['quality_score'].mean(), color='black',
            linestyle='-', linewidth=1.5, label=f"Mean = {df_known['quality_score'].mean():.3f}")

plt.title('Quality Score Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Quality Score')
plt.ylabel('Frequency (Days)')
plt.legend()
plt.tight_layout()
plt.savefig('../results/plots/qs_distribution.png', dpi=300)
plt.show()

### Plot 5 — Signal Count Per Year (Stacked Bar)

In [ ]:
# Add signal column (if not already created)
def classify_qs(score):
    if score >= 0.6:   return 'Bullish'
    elif score <= 0.4: return 'Bearish'
    else:              return 'Neutral'

df_known['signal'] = df_known['quality_score'].apply(classify_qs)

# Now run Plot 6
signal_counts = df_known.groupby(['year', 'signal']).size().unstack(fill_value=0)

# Reorder columns consistently
signal_counts = signal_counts.reindex(columns=['Bearish', 'Neutral', 'Bullish'], fill_value=0)

signal_counts.plot(
    kind='bar', stacked=True, figsize=(8, 5),
    color=['red', 'orange', 'green'],
    edgecolor='white'
)

plt.title('Bullish / Neutral / Bearish Days Per Year', fontsize=14, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Number of Trading Days')
plt.xticks(rotation=0)
plt.legend(title='Signal')
plt.tight_layout()
plt.savefig('../results/plots/qs_signal_counts.png', dpi=300)
plt.show()

In [ ]:
#avg quality score 
avg_scores = df_known['quality_score'].mean().round(2)

In [ ]:
def save_company_score(company_name, avg_score, filename='../results/company_scores.csv'):
    # Generate the signal first so we can use it in the dict and the print
    signal = classify_qs(avg_score)
    
    new_entry = pd.DataFrame([{
        'company': company_name, 
        'quality score': avg_score, 
        'signal': signal
    }])

    if os.path.isfile(filename):
        df = pd.read_csv(filename)
        # Combine and keep the newest entry for that company
        df = pd.concat([df, new_entry]).drop_duplicates(subset=['company'], keep='last')
    else:
        df = new_entry

    df.to_csv(filename, index=False)
    print(f"Logged: {company_name} | Score: {avg_score} | Signal: {signal}")

save_company_score(name, avg_scores)